# M4 — Benchmark 2025 multivariável (EF01): M1+M2+M3 × ano intocado (só inferência)

Passo 3 do `multivariavel/PLANO.md`: os 27 checkpoints (M1-DLinear-multi,
M2-PatchTST-CI, M3-PatchTST-CD; 3 H × 3 seeds) previstos no ano de 2025, que é o
**primeiro toque** — nenhum treino, val, tuning, seleção ou ajuste de peso em 2025.

Regras de honestidade (invioláveis, travadas em código na §8):
- `torch.set_grad_enabled(False)` global + `@torch.no_grad()` em toda previsão +
  modelos sempre `.eval()`; nenhum otimizador é instanciado (varredura do próprio
  `.ipynb` na §8 trava isso).
- Limpeza idêntica ao M1 (interp `time` limite 24 por canal, descarte conjunto,
  winsorize da turbidez no p99 train-only, z-stats dos `normalizacao.json` de cada
  experimento — nada é recalculado em 2025 além de aplicar).
- Agregação por (modelo, H) = média das 3 seeds (pesos 1/3 fixos); ensemble de média
  simples M1+M2+M3 só como diagnóstico (PLANO §2: único ensemble final permitido).
- Pisos `sazonal-naive-288` por (H, canal) + `persistencia` como contexto.
- Janelas `L=2304 → H∈{12,72,288}` rolantes no ano 2025 inteiro; cobertura por H,
  dias-âncora 23:55 e quebra mensal (espelho do 18-v2).

In [1]:
import gc
import json
import os
import socket
import time
import warnings
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from numpy.lib.stride_tricks import sliding_window_view
from sklearn.metrics import mean_absolute_error, mean_squared_error

import torch
import torch.nn as nn

warnings.filterwarnings("ignore")
plt.rcParams.update({"figure.dpi": 110})

# TRAVA DE HONESTIDADE (regras do M4): 2025 = SÓ inferência. Gradiente desligado
# no processo inteiro; toda previsão roda sob torch.no_grad com modelo .eval().
torch.set_grad_enabled(False)
assert not torch.is_grad_enabled(), "trava global de gradiente inativa!"

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "multivariavel" / "dados" / "treino").exists())
OUT = ROOT / "multivariavel" / "resultados" / "M4-benchmark-2025"
(OUT / "figs").mkdir(parents=True, exist_ok=True)

L = 2304                      # 8 d de contexto (passo 5 min), verbatim M1
HS = [12, 72, 288]            # 1 h / 6 h / 24 h — um pipeline dedicado por H
SEASON = 288
INTERP_LIMIT = 24             # 2 h, verbatim M1
CH = ["od", "ph", "temp", "turb"]             # precipitação EXCLUÍDA (PLANO §1)
SEEDS = [42, 7, 123]          # 3 seeds por (config, H) = 27 checkpoints
FAMS = ["m1", "m2", "m3"]     # m1=dlinear-multi, m2=patchtst-CI, m3=patchtst-CD
FAM_NOME = {"m1": "dlinear-multi", "m2": "patchtst-CI", "m3": "patchtst-CD"}
CKPT_DIR = {"m1": ROOT / "multivariavel" / "resultados" / "M1-dlinear-multi" / "modelos",
            "m2": ROOT / "multivariavel" / "resultados" / "M2-patchtst-multi-CI" / "modelos",
            "m3": ROOT / "multivariavel" / "resultados" / "M3-patchtst-multi-CD" / "modelos"}
CKPT_PAT = {"m1": "dlinear_multi_H%d_s%d.pt", "m2": "patchtst_CI_H%d_s%d.pt",
            "m3": "patchtst_CD_H%d_s%d.pt"}
# Réguas uni-v2 H=288 no mesmo ano de teste (18-v2-benchmark-2025, comparação direta
# de ano; caveat: val de origem distinta). Só contexto honesto, sem declarar régua nova.
REGUA_UNI = {"ph": 0.0465, "od": 0.2056}

_raw_dev = os.environ.get("M4_DEVICE", "cuda" if torch.cuda.is_available() else "cpu")
try:
    DEVICE = torch.device(_raw_dev)
    torch.zeros(1).to(DEVICE)
except Exception as e:
    print("M4_DEVICE=" + str(_raw_dev) + " indisponível (" + str(e) + ") -> fallback CPU")
    DEVICE = torch.device("cpu")

print("ROOT:", ROOT, "| torch:", torch.__version__, "| DEVICE:", DEVICE)
if DEVICE.type == "cuda":
    print("gpu:", torch.cuda.get_device_name(0))
print("host:", socket.gethostname(), "| cpu:", os.cpu_count(),
      "| L:", L, "| Hs:", HS, "| seeds:", SEEDS)
print("threads:", {k: os.environ.get(k, "<unset>") for k in
      ("OMP_NUM_THREADS", "MKL_NUM_THREADS", "OPENBLAS_NUM_THREADS",
       "M4_DEVICE", "CUDA_VISIBLE_DEVICES")})
print("OUT:", OUT)
print("grad global ativo?", torch.is_grad_enabled(), "(exigido: False)")
t_wall0 = time.time()

ckpts = [CKPT_DIR[f] / (CKPT_PAT[f] % (H, sd)) for f in FAMS for H in HS for sd in SEEDS]
faltam = [str(p) for p in ckpts if not p.exists()]
assert not faltam, "checkpoints ausentes: %s" % faltam[:5]
print("27 checkpoints M1/M2/M3 OK")

ROOT: /home/administrador/Projects/temporal-model-prediction | torch: 2.14.0+cu126 | DEVICE: cuda
gpu: NVIDIA RTX 4000 Ada Generation
host: administrador-HP-Z4-G5-Workstation-Desktop-PC | cpu: 20 | L: 2304 | Hs: [12, 72, 288] | seeds: [42, 7, 123]
threads: {'OMP_NUM_THREADS': '<unset>', 'MKL_NUM_THREADS': '<unset>', 'OPENBLAS_NUM_THREADS': '<unset>', 'M4_DEVICE': 'cuda', 'CUDA_VISIBLE_DEVICES': '0'}
OUT: /home/administrador/Projects/temporal-model-prediction/multivariavel/resultados/M4-benchmark-2025
grad global ativo? False (exigido: False)
27 checkpoints M1/M2/M3 OK


## 1. Carga 2025 — primeiro toque

Um CSV `multivariavel/dados/benchmark/`, parse CETESB (`;`, decimal vírgula,
`windows-1252`, pula linha 1, `dd/mm/aaaa hh:mm`), reindex na grade anual cheia de
2025. Colunas OD + pH + Temp + Turb (precipitação excluída, PLANO §1).

In [2]:
REN = {"Data hora": "ds", "Oxigênio Dissolvido (mg/L)": "od", "pH": "ph",
       "Temperatura (°C)": "temp", "Turbidez (NTU)": "turb"}
csv25 = ROOT / "multivariavel" / "dados" / "benchmark" / "ef01-mogi-das-cruzes_multivariavel_2025.csv"
df25 = pd.read_csv(csv25, sep=";", decimal=",", encoding="windows-1252", skiprows=1,
                   parse_dates=["Data hora"], dayfirst=True, na_values=[""])
assert "Precipitação (mm)" in df25.columns, "coluna de precipitação sumiu do CSV!"
df25 = df25.rename(columns=REN)[["ds", "od", "ph", "temp", "turb"]].sort_values("ds").reset_index(drop=True)
print("linhas CSV:", len(df25), "|", df25["ds"].min(), "->", df25["ds"].max())
print("NaN internos do CSV (linhas lidas):", df25[CH].isna().sum().to_dict())
idx25 = pd.date_range("2025-01-01", "2025-12-31 23:55", freq="5min")  # grade anual cheia
s_raw = df25.set_index("ds")[CH].reindex(idx25)
print("grade 2025:", len(s_raw), "| esperado:", 365 * 288)
assert len(s_raw) == 365 * 288 == 105120, len(s_raw)
dt = np.diff(s_raw.index.values.astype("datetime64[m]").astype(np.int64))
assert (dt == 5).all(), "grade não-uniforme!"
print("NaN pré-interp (grade cheia):", s_raw.isna().sum().to_dict(),
      {c: round(100 * float(s_raw[c].isna().sum()) / len(s_raw), 2) for c in CH})
print("diferença grade-cheia menos CSV = cauda 31/dez ausente no arquivo:",
      {c: int(s_raw[c].isna().sum() - df25[c].isna().sum()) for c in CH})
assert list(s_raw.columns) == CH and "Precipitação (mm)" not in s_raw.columns
print("grade 5min uniforme 2025 OK | precipitação excluída OK")
print(s_raw.describe().round(3).to_string())

linhas CSV: 104833 | 2025-01-01 00:00:00 -> 2025-12-31 00:00:00
NaN internos do CSV (linhas lidas): {'od': 387, 'ph': 6093, 'temp': 221, 'turb': 2979}
grade 2025: 105120 | esperado: 105120
NaN pré-interp (grade cheia): {'od': 674, 'ph': 6380, 'temp': 508, 'turb': 3266} {'od': 0.64, 'ph': 6.07, 'temp': 0.48, 'turb': 3.11}
diferença grade-cheia menos CSV = cauda 31/dez ausente no arquivo: {'od': 287, 'ph': 287, 'temp': 287, 'turb': 287}
grade 5min uniforme 2025 OK | precipitação excluída OK
               od         ph        temp        turb
count  104446.000  98740.000  104612.000  101854.000
mean        5.466      6.196      20.893      12.558
std         1.414      0.180       2.788      16.685
min         1.740      5.590      14.610       1.810
25%         4.410      6.070      18.910       6.400
50%         5.590      6.190      20.930       8.030
75%         6.560      6.330      23.000      11.140
max         8.700      6.670      28.010     238.100


## 2. EDA 2025 (+ estação austral — SÓ reporte, nunca feature)

In [3]:
v_raw = s_raw.to_numpy()
for j, c in enumerate(CH):
    isna = np.isnan(v_raw[:, j])
    gaps = np.diff(np.concatenate([[0], np.where(~isna)[0], [len(isna)]])) - 1
    print("%s: faltantes=%d (%.2f%%) | maior gap=%.1f h | blocos=%d" % (
        c, int(isna.sum()), 100 * isna.mean(), gaps.max() * 5 / 60, int((gaps > 0).sum())))
print("blocos NaN por canal (até 15):")
for c in CH:
    gi = np.where(s_raw[c].isna().to_numpy())[0]
    blocos = np.split(gi, np.where(np.diff(gi) > 1)[0] + 1) if len(gi) else []
    for g in blocos[:15]:
        print("  %s outage %s -> %s (%d slots = %.1f h)" % (
            c, s_raw.index[g[0]], s_raw.index[g[-1]], len(g), len(g) * 5 / 60))
    if len(blocos) > 15:
        print("  %s ... +%d blocos" % (c, len(blocos) - 15))
print("NaN por mês (grade cheia):")
print(s_raw.isna().groupby(s_raw.index.month).sum().to_string())

fig, ax = plt.subplots(4, 1, figsize=(12, 9), sharex=True)
for a, c in zip(ax, CH):
    a.plot(s_raw.index, s_raw[c].values, lw=0.4)
    a.set_title("%s 2025 — benchmark (ano intocado até aqui; n=%d faltantes)" % (
        c, int(s_raw[c].isna().sum())))
fig.tight_layout(); fig.savefig(OUT / "figs" / "01-eda.png"); plt.close(fig)
print("fig 01-eda salva")

od: faltantes=674 (0.64%) | maior gap=23.9 h | blocos=41
ph: faltantes=6380 (6.07%) | maior gap=23.9 h | blocos=3361
temp: faltantes=508 (0.48%) | maior gap=23.9 h | blocos=20
turb: faltantes=3266 (3.11%) | maior gap=23.9 h | blocos=2119
blocos NaN por canal (até 15):
  od outage 2025-02-05 09:35:00 -> 2025-02-05 12:40:00 (38 slots = 3.2 h)
  od outage 2025-03-06 17:45:00 -> 2025-03-06 17:55:00 (3 slots = 0.2 h)
  od outage 2025-03-17 10:00:00 -> 2025-03-17 10:00:00 (1 slots = 0.1 h)
  od outage 2025-03-17 10:10:00 -> 2025-03-17 13:00:00 (35 slots = 2.9 h)
  od outage 2025-04-02 09:15:00 -> 2025-04-02 14:25:00 (63 slots = 5.2 h)
  od outage 2025-04-21 16:30:00 -> 2025-04-21 16:30:00 (1 slots = 0.1 h)
  od outage 2025-05-13 07:10:00 -> 2025-05-13 07:15:00 (2 slots = 0.2 h)
  od outage 2025-05-14 15:40:00 -> 2025-05-14 15:45:00 (2 slots = 0.2 h)
  od outage 2025-05-17 13:30:00 -> 2025-05-17 14:00:00 (7 slots = 0.6 h)
  od outage 2025-05-17 15:05:00 -> 2025-05-17 15:05:00 (1 slots = 0.1 h

fig 01-eda salva


## 3. Limpeza idêntica ao M1

Interp `time` limite 24 **por canal**; descarte **conjunto** (qualquer dos 4 canais
com NaN mata a janela, §4). Winsorize da turbidez no p99 **train-only** e z-stats
dos `normalizacao.json` de cada experimento — aplicados, nunca recalculados em 2025.

In [4]:
NJ = {f: json.load(open(CKPT_DIR[f] / "normalizacao.json")) for f in FAMS}
for f in FAMS:
    assert NJ[f]["channels"] == CH and NJ[f]["L"] == L and NJ[f]["Hs"] == HS
    assert NJ[f]["seeds"] == SEEDS, NJ[f]["seeds"]
P99 = {f: NJ[f]["turb_winsor_p99_train_only_H288"] for f in FAMS}
print("turb p99 train-only por experimento:", P99)
assert max(P99.values()) - min(P99.values()) < 1e-6, P99
TURB_P99 = P99["m1"]
MU = {f: {H: np.array(NJ[f]["per_H"][str(H)]["mu"], dtype=np.float64) for H in HS} for f in FAMS}
SD = {f: {H: np.array(NJ[f]["per_H"][str(H)]["sd"], dtype=np.float64) for H in HS} for f in FAMS}
dmax = max(float(np.abs(MU[a][H] - MU[b][H]).max()) + float(np.abs(SD[a][H] - SD[b][H]).max())
           for a in FAMS for b in FAMS for H in HS)
print("divergência máx de z-stats entre experimentos: %.2e (exigido ~0)" % dmax)
assert dmax < 1e-6, "z-stats divergem entre M1/M2/M3!"

s = s_raw.interpolate(method="time", limit=INTERP_LIMIT)
pos = s.isna().sum().to_dict()
print("2025 pós-interp por canal:", pos)
jd = int(s.isna().any(axis=1).sum())
print("descarte conjunto 2025: %d slots (%.2f%% da grade)" % (jd, 100 * jd / len(s)))

amostra = slice("2025-09-09", "2025-09-16")
fig, ax = plt.subplots(4, 1, figsize=(12, 9), sharex=True)
for a, c in zip(ax, CH):
    a.plot(s_raw[c][amostra].index, s_raw[c][amostra].values, ".", ms=2, label="cru")
    a.plot(s[c][amostra].index, s[c][amostra].values, lw=0.8, label="interp lim24")
    a.set_title(c); a.legend(fontsize=8)
fig.tight_layout(); fig.savefig(OUT / "figs" / "02-limpeza.png"); plt.close(fig)
print("fig 02-limpeza salva")

turb p99 train-only por experimento: {'m1': 59.220001220703125, 'm2': 59.220001220703125, 'm3': 59.220001220703125}
divergência máx de z-stats entre experimentos: 0.00e+00 (exigido ~0)
2025 pós-interp por canal: {'od': 392, 'ph': 807, 'temp': 351, 'turb': 352}
descarte conjunto 2025: 845 slots (0.80% da grade)


fig 02-limpeza salva


## 4. Janelamento rolante (`L=2304`, um pipeline por H) + dias-âncora 23:55

Janelas por data de **fim** no ano 2025 inteiro; válida = sem NaN pós-interp em
**nenhum dos 4 canais** no span `L+H` (mesma semântica do M1, sem purge — sem treino
não há vazamento a purgar). Cobertura reportada por H.

In [5]:
V = s.to_numpy().astype(np.float64)          # (N, 4) pós-interp; NaN = outage real
IDX = s.index
N = len(s)

def valid_starts(nan4, W):
    """Starts s com zero NaN em nan4[s:s+W] (cumsum; mesma semântica do M1)."""
    c = np.zeros((nan4.shape[0] + 1, nan4.shape[1]), dtype=np.int32)
    np.cumsum(nan4.astype(np.int32), axis=0, out=c[1:])
    return ((c[W:] - c[:-W]) == 0).all(axis=1)

NAN4 = np.isnan(V)
P = {}
for H in HS:
    W = L + H
    ok = valid_starts(NAN4, W)
    S = np.where(ok)[0]
    ends = IDX[S + W - 1]
    am = np.where(ends.time == pd.Timestamp("23:55").time())[0]
    ex = [0, len(am) // 2, len(am) - 1] if len(am) >= 3 else list(range(len(am)))
    P[H] = {"W": W, "starts": S, "ends": ends, "n": len(S),
            "anchors": am, "examples": [int(am[k]) for k in ex]}
    print("H=%3d janelas viáveis=%d/%d (%.1f%%) | dias-âncora 23:55=%d (%s -> %s)" % (
        H, len(S), N - W + 1, 100 * len(S) / (N - W + 1), len(am),
        ends[am[0]].date(), ends[am[-1]].date()))
    assert len(am) >= 200, "âncoras insuficientes em H=%d: %d" % (H, len(am))
print("dias-exemplo por H:", {H: [str(P[H]['ends'][k].date()) for k in P[H]['examples']] for H in HS})

H= 12 janelas viáveis=74752/102805 (72.7%) | dias-âncora 23:55=261 (2025-01-09 -> 2025-12-30)
H= 72 janelas viáveis=74032/102745 (72.1%) | dias-âncora 23:55=260 (2025-01-09 -> 2025-12-30)
H=288 janelas viáveis=71607/102529 (69.8%) | dias-âncora 23:55=250 (2025-01-09 -> 2025-12-30)
dias-exemplo por H: {12: ['2025-01-09', '2025-07-13', '2025-12-30'], 72: ['2025-01-09', '2025-07-14', '2025-12-30'], 288: ['2025-01-09', '2025-07-14', '2025-12-30']}


## 5. Pisos por (H, canal): `sazonal-naive-288`

Piso = cópia do dia anterior no mesmo horário (`V[s+L−288 : s+L−288+H]` por canal);
para H<288 é o **prefixo** dessa cópia. `persistencia` entra só como contexto.

In [6]:
V32 = V.astype(np.float32)

def floor_chunk(S, H):
    """Pisos (C,H,4) em unidade original p/ starts S do pipeline H (verbatim M1 §6)."""
    S = np.asarray(S)
    b = S + L - SEASON
    idx = b[:, None] + np.arange(H)[None, :]
    saz = V32[idx]                                   # cópia do dia anterior
    per = np.repeat(V32[S + L - 1][:, None, :], H, axis=1)
    return {"sazonal-naive-288": saz, "persistencia": per}

def true_chunk(S, H):
    S = np.asarray(S)
    idx = (S + L)[:, None] + np.arange(H)[None, :]
    return V32[idx]                                  # (C,H,4) unidade original

for H in HS:
    Y0 = true_chunk(P[H]["starts"][:8], H)
    F0 = floor_chunk(P[H]["starts"][:8], H)
    assert Y0.shape == (8, H, 4) and F0["sazonal-naive-288"].shape == (8, H, 4)
    assert np.isfinite(Y0).all() and np.isfinite(F0["sazonal-naive-288"]).all()
    print("H=%3d sanity pisos OK (janelas viáveis não têm NaN no span L+H)" % H)
del Y0, F0

H= 12 sanity pisos OK (janelas viáveis não têm NaN no span L+H)
H= 72 sanity pisos OK (janelas viáveis não têm NaN no span L+H)
H=288 sanity pisos OK (janelas viáveis não têm NaN no span L+H)


## 6. Time-features + winsorize da turbidez + normalização train-only

Time-features (vocabulário 12/16, determinísticas do timestamp): `tod_sin/cos` +
`solar/90` por passo + 4 Fourier da origem. Winsorize e z-score **aplicados** com os
números do treino — nada fitado em 2025.

In [7]:
LAT, LON, TZ = -23.52, -46.19, -3  # Mogi das Cruzes (verbatim 12/16)

def elevacao_solar(ts, lat=LAT, lon=LON, tz=TZ):
    ts = pd.DatetimeIndex(ts)
    doy = ts.dayofyear.to_numpy() + (ts.hour.to_numpy() + ts.minute.to_numpy() / 60) / 24
    g = 2 * np.pi / 365 * (doy - 1 + (ts.hour.to_numpy() - 12) / 24)
    eq = 229.18 * (0.000075 + 0.001868 * np.cos(g) - 0.032077 * np.sin(g)
                   - 0.014615 * np.cos(2 * g) - 0.040849 * np.sin(2 * g))
    decl = (0.006918 - 0.399912 * np.cos(g) + 0.070257 * np.sin(g) - 0.006758 * np.cos(2 * g)
            + 0.000907 * np.sin(2 * g) - 0.002697 * np.cos(3 * g) + 0.00148 * np.sin(3 * g))
    tst = (ts.hour.to_numpy() * 60 + ts.minute.to_numpy()) + eq + 4 * lon - 60 * tz
    ha = np.radians(tst / 4 - 180)
    cosz = np.sin(np.radians(lat)) * np.sin(decl) + np.cos(np.radians(lat)) * np.cos(decl) * np.cos(ha)
    return 90 - np.degrees(np.arccos(np.clip(cosz, -1, 1)))

def fourier_doy(ts, n=366):
    d = pd.DatetimeIndex(ts).dayofyear.to_numpy()
    return (np.sin(2 * np.pi * d / n), np.cos(2 * np.pi * d / n),
            np.sin(4 * np.pi * d / n), np.cos(4 * np.pi * d / n))

TOD_SIN = np.sin(2 * np.pi * (IDX.hour.to_numpy() * 60 + IDX.minute.to_numpy()) / 1440.0).astype(np.float32)
TOD_COS = np.cos(2 * np.pi * (IDX.hour.to_numpy() * 60 + IDX.minute.to_numpy()) / 1440.0).astype(np.float32)
SOLAR = (elevacao_solar(IDX) / 90.0).astype(np.float32)
F3 = np.stack([TOD_SIN, TOD_COS, SOLAR], axis=1)  # (N, 3) feats por passo
assert F3.shape == (N, 3)

# Aplica winsorize (threshold do treino) e z-stats (do treino) por (família, H)
Vclip = V32.copy()
Vclip[:, 3] = np.minimum(Vclip[:, 3], TURB_P99)
print("winsorize 2025 aplicado em p99=%.2f NTU: fração de slots clipados = %.4f" % (
    TURB_P99, float((V32[:, 3] > TURB_P99).mean())))
Vz = {f: {H: ((Vclip - MU[f][H]) / SD[f][H]).astype(np.float32) for H in HS} for f in FAMS}
for f in FAMS:
    for H in HS:
        assert np.isfinite(Vz[f][H][~NAN4.any(axis=1)]).all(), (f, H)
print("Vz por (família, H) OK — finita em todo slot limpo (janelas viáveis garantidas no §4)")

winsorize 2025 aplicado em p99=59.22 NTU: fração de slots clipados = 0.0234
Vz por (família, H) OK — finita em todo slot limpo (janelas viáveis garantidas no §4)


## 7. Arquiteturas verbatim (cópias fiéis M1/M2/M3) — SÓ inferência

`DLinearMulti` (M1) · `PatchTSTCI` (M2, pesos compartilhados, forward por canal) ·
`PatchTST_CD` (M3, joint-attention). Nenhum treino: este notebook não instancia
otimizador nem agenda — só carrega `state_dict` e prevê.

In [8]:
POOL_K = 25
LN, PATCH_P, PATCH_S = 2016, 48, 24          # 83 tokens CI ((2016-48)//24+1); CD usa L=2304 -> 95
D_MODEL, NLAYERS, NHEAD, FF, DROPOUT = 64, 3, 4, 128, 0.1
N_TF = 7                      # time-feats por canal no CI: tod_sin/cos + solar + 4 Fourier-origem
DIN = 11

class DLinearMulti(nn.Module):
    """Cópia fiel do M1 §8: 11 séries -> pool k=25 -> lineares por canal-alvo -> 2 heads."""
    def __init__(self, din=DIN, Lin=L, H=288, k=POOL_K):
        super().__init__()
        self.pool = nn.AvgPool1d(k, stride=1, padding=k // 2)
        self.linT_ph = nn.Linear(din * Lin, H)
        self.linS_ph = nn.Linear(din * Lin, H)
        self.linT_od = nn.Linear(din * Lin, H)
        self.linS_od = nn.Linear(din * Lin, H)
        self.gamma = nn.Parameter(torch.ones(4))
        self.beta = nn.Parameter(torch.zeros(4))
    def forward(self, x):
        v = x[:, :4, :]
        mu = v.mean(dim=2, keepdim=True); sg = v.std(dim=2, keepdim=True).clamp_min(1e-3)
        g = self.gamma[None, :, None]; b = self.beta[None, :, None]
        xn = torch.cat([g * (v - mu) / sg + b, x[:, 4:, :]], dim=1)
        t = self.pool(xn); s = xn - t
        tf, sf = t.flatten(1), s.flatten(1)
        yph = self.linT_ph(tf) + self.linS_ph(sf)
        yod = self.linT_od(tf) + self.linS_od(sf)
        yph = (yph - self.beta[1]) / self.gamma[1].clamp_min(1e-3) * sg[:, 1] + mu[:, 1]
        yod = (yod - self.beta[0]) / self.gamma[0].clamp_min(1e-3) * sg[:, 0] + mu[:, 0]
        return torch.stack([yph, yod], dim=2)

class PatchTSTCI(nn.Module):
    """Cópia fiel do M2 §8: mesmos pesos, forward separado por canal (CI estrito)."""
    def __init__(self, H, n_tf=N_TF):
        super().__init__()
        self.n_tf = n_tf
        self.N = (LN - PATCH_P) // PATCH_S + 1
        assert self.N == 83, self.N
        self.proj = nn.Linear((1 + n_tf) * PATCH_P, D_MODEL)
        self.pos = nn.Parameter(torch.randn(1, self.N, D_MODEL) * 0.02)
        layer = nn.TransformerEncoderLayer(D_MODEL, NHEAD, FF, DROPOUT, batch_first=True)
        self.enc = nn.TransformerEncoder(layer, NLAYERS)
        self.drop = nn.Dropout(DROPOUT)
        self.head = nn.Linear(self.N * D_MODEL, H)
        self.gamma = nn.Parameter(torch.ones(1))
        self.beta = nn.Parameter(torch.zeros(1))
    def forward(self, xc):
        v = xc[:, :1, :]
        mu = v.mean(dim=2, keepdim=True)
        sg = v.std(dim=2, keepdim=True).clamp_min(1e-3)
        xn = torch.cat([self.gamma * (v - mu) / sg + self.beta, xc[:, 1:, :]], dim=1)
        w = xn.unfold(2, PATCH_P, PATCH_S).permute(0, 2, 1, 3).reshape(xc.size(0), self.N, -1)
        z = self.proj(w) + self.pos
        z = self.enc(self.drop(z))
        y = self.head(self.drop(z.flatten(1)))
        return (y - self.beta) / self.gamma.clamp_min(1e-3) * sg[:, :, 0] + mu[:, :, 0]

class PatchTST_CD(nn.Module):
    """Cópia fiel do M3 §8: patches 48/24 sobre as 11 séries empilhadas (joint-attention)."""
    def __init__(self, din=DIN, Lin=L, H=288):
        super().__init__()
        self.N = (Lin - PATCH_P) // PATCH_S + 1
        self.proj = nn.Linear(din * PATCH_P, D_MODEL)
        self.pos = nn.Parameter(torch.randn(1, self.N, D_MODEL) * 0.02)
        layer = nn.TransformerEncoderLayer(D_MODEL, NHEAD, FF, DROPOUT, batch_first=True)
        self.enc = nn.TransformerEncoder(layer, NLAYERS)
        self.drop = nn.Dropout(DROPOUT)
        self.head_ph = nn.Linear(self.N * D_MODEL, H)
        self.head_od = nn.Linear(self.N * D_MODEL, H)
        self.gamma = nn.Parameter(torch.ones(4))
        self.beta = nn.Parameter(torch.zeros(4))
    def forward(self, x):
        v = x[:, :4, :]
        mu = v.mean(dim=2, keepdim=True); sg = v.std(dim=2, keepdim=True).clamp_min(1e-3)
        g = self.gamma[None, :, None]; b = self.beta[None, :, None]
        xn = torch.cat([g * (v - mu) / sg + b, x[:, 4:, :]], dim=1)
        z = self.proj(xn.unfold(2, PATCH_P, PATCH_S).permute(0, 2, 1, 3).flatten(2)) + self.pos
        z = self.enc(self.drop(z))
        f = self.drop(z.flatten(1))
        yph = self.head_ph(f)
        yod = self.head_od(f)
        yph = (yph - self.beta[1]) / self.gamma[1].clamp_min(1e-3) * sg[:, 1] + mu[:, 1]
        yod = (yod - self.beta[0]) / self.gamma[0].clamp_min(1e-3) * sg[:, 0] + mu[:, 0]
        return torch.stack([yph, yod], dim=2)

for H in HS:
    print("H=%3d params: DLinear=%d | CI=%d | CD=%d" % (
        H, sum(p.numel() for p in DLinearMulti(H=H).parameters()),
        sum(p.numel() for p in PatchTSTCI(H=H).parameters()),
        sum(p.numel() for p in PatchTST_CD(H=H).parameters())))

H= 12 params: DLinear=1216568 | CI=194126 | CD=286304


H= 72 params: DLinear=7299368 | CI=512906 | CD=1016024


H=288 params: DLinear=29197448 | CI=1660514 | CD=3643016


## 8. Inferência seed-mean + média-simples-diagnóstico + trava executável

Por (família, H): 3 seeds → média simples (pesos 1/3 fixos). Diagnóstico final =
média simples das 3 seed-means (único ensemble permitido, PLANO §2). A trava abaixo
garante: gradiente global desligado, `.eval()` em todo modelo carregado e ausência
de tokens de treino no próprio `.ipynb`.

In [9]:
FLN = sliding_window_view(F3, LN, axis=0)  # view (N-LN+1, 3, LN), verbatim M2

def stack4(xv, xt):
    """Empilha os 4 canais no batch: (B,4,LN)+(B,7,LN) -> (B*4, 8, LN), ordem od,ph,temp,turb."""
    B = xv.size(0)
    return torch.cat([xv.permute(1, 0, 2).reshape(B * 4, 1, LN), xt.repeat(4, 1, 1)], dim=1)

def monta(H, pos, fam):
    """Monta (X (B,11,L), Y (B,H,2)[ph,od] z-score) p/ posições pos do pipeline H (verbatim M1/M3)."""
    ii = np.asarray(pos)
    S = P[H]["starts"][ii]
    WL = sliding_window_view(Vz[fam][H], L, axis=0)
    FL = sliding_window_view(F3, L, axis=0)
    X = np.concatenate([WL[S], FL[S]], axis=1)
    E = P[H]["ends"][ii]
    forg = np.column_stack([a.astype(np.float32) for a in fourier_doy(E)])
    X = np.concatenate([X, np.repeat(forg[:, :, None], L, axis=2)], axis=1)
    Tz = Vz[fam][H][:, [1, 0]]
    WY = sliding_window_view(Tz, H, axis=0)
    return X.astype(np.float32), WY[S + L].transpose(0, 2, 1).astype(np.float32)

def monta_ci(H, pos, WLN_H, WY_H):
    """Monta (Xv (B,4,LN), Xt (B,7,LN)) p/ posições pos do pipeline H (verbatim M2)."""
    ii = np.asarray(pos)
    Sb = P[H]["starts"][ii]
    r = Sb + L - LN
    Xv = WLN_H[r]
    Xt = np.concatenate([FLN[r],
                         np.repeat(np.column_stack(
                             [a.astype(np.float32) for a in fourier_doy(P[H]["ends"][ii])])[:, :, None],
                             LN, axis=2)], axis=1)
    return Xv.astype(np.float32), Xt.astype(np.float32)

MODELS = {}   # (fam, H, seed) -> módulo em DEVICE, sempre .eval()
EVAL_OK = []    # um True por modelo carregado em .eval() (trava §8 confere)
BATCH_F = {"m1": {12: 2048, 72: 2048, 288: 1024},
           "m2": {12: 1024, 72: 1024, 288: 512},
           "m3": {12: 2048, 72: 2048, 288: 1024}}

def _carrega(fam, H, seed):
    key = (fam, H, seed)
    if key not in MODELS:
        cls = {"m1": DLinearMulti, "m2": PatchTSTCI, "m3": PatchTST_CD}[fam]
        m = cls(H=H).to(DEVICE)
        ckpt = torch.load(CKPT_DIR[fam] / (CKPT_PAT[fam] % (H, seed)),
                          map_location=DEVICE, weights_only=False)
        m.load_state_dict(ckpt["state"])
        m.eval()
        assert not m.training, "modelo fora de .eval()!"
        EVAL_OK.append(True)
        MODELS[key] = m
    return MODELS[key]

@torch.no_grad()
def preve_fam(fam, H, pos, WLN_H=None):
    """Seed-mean (C,H,2)[ph,od] em UNIDADE ORIGINAL + lista por seed (só inferência)."""
    ii = np.asarray(pos)
    mu, sd = MU[fam][H], SD[fam][H]
    acc, por_seed = None, []
    for seed in SEEDS:
        m = _carrega(fam, H, seed)
        outs = []
        for b in range(0, len(ii), BATCH_F[fam][H]):
            sub = ii[b:b + BATCH_F[fam][H]]
            if fam == "m2":
                Xv, Xt = monta_ci(H, sub, WLN_H, None)
                Xin = stack4(torch.from_numpy(Xv), torch.from_numpy(Xt)).to(DEVICE)
                Z = m(Xin).cpu().numpy().reshape(4, len(sub), H).transpose(1, 2, 0)
                Zn = Z * sd + mu     # broadcast por canal, ordem CH=[od,ph,temp,turb]
                outs.append(np.stack([Zn[:, :, 1], Zn[:, :, 0]], axis=2))
            else:
                Xb, _ = monta(H, sub, fam)
                Z = m(torch.from_numpy(Xb).to(DEVICE)).cpu().numpy()
                outs.append(np.stack([Z[:, :, 0] * sd[1] + mu[1],
                                      Z[:, :, 1] * sd[0] + mu[0]], axis=2))
        Ps = np.concatenate(outs)
        assert np.isfinite(Ps).all(), (fam, H, seed)
        por_seed.append(Ps)
        acc = Ps if acc is None else acc + Ps
    return (acc / len(SEEDS)), por_seed

def _descarrega(fam, H):
    for seed in SEEDS:
        MODELS.pop((fam, H, seed), None)
    gc.collect()
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()

# Sanity rápido: 4 janelas em H=12 por família (pesa pouco, valida os 3 pipelines)
for _fam in FAMS:
    _WLN = sliding_window_view(Vz[_fam][12], LN, axis=0) if _fam == "m2" else None
    _sm, _ps = preve_fam(_fam, 12, np.arange(4), _WLN)
    assert _sm.shape == (4, 12, 2) and all(p.shape == (4, 12, 2) for p in _ps)
    _descarrega(_fam, 12)
    print("sanity %s H=12 OK (seed-mean %s, finita)" % (_fam, _sm.shape))
del _fam, _WLN, _sm, _ps

sanity m1 H=12 OK (seed-mean (4, 12, 2), finita)


sanity m2 H=12 OK (seed-mean (4, 12, 2), finita)
sanity m3 H=12 OK (seed-mean (4, 12, 2), finita)


In [10]:
# Trava executável de honestidade: (i) gradiente global desligado; (ii) todo modelo
# carregado entrou em .eval() (EVAL_OK); (iii) varredura do próprio .ipynb contra
# tokens de treino (construídos por concatenação p/ a varredura não se auto-acusar).
assert not torch.is_grad_enabled(), "gradiente global religado — ABORTAR"
assert len(EVAL_OK) >= 3 and all(EVAL_OK), "há modelo fora de .eval()!"
assert all(not m.training for m in MODELS.values()), "há modelo fora de .eval()!"
_src = "".join(open(__file__ if "__file__" in dir() else
                    str(next(p for p in [Path.cwd(), *Path.cwd().parents]
                             if (p / "multivariavel" / "notebooks" / "M4-benchmark-2025.ipynb").exists())
                        / "multivariavel" / "notebooks" / "M4-benchmark-2025.ipynb")).read())
_n_eval = _src.count(".eva" + "l()")
_proib = ["torch" + ".opt" + "im", ".tra" + "in(", "back" + "ward",
          "Early" + "Stopping", "requires" + "_grad", "zero" + "_grad", ".st" + "ep("]
_ach = [t for t in _proib if t in _src]
print("ocorrências de .eval() no notebook:", _n_eval, "(exigido >= 3)")
print("tokens de treino no notebook:", _ach if _ach else "nenhum")
assert _n_eval >= 3 and not _ach, "trava de honestidade falhou!"
print("trava de honestidade VERDE: só inferência neste notebook")

ocorrências de .eval() no notebook: 11 (exigido >= 3)
tokens de treino no notebook: nenhum
trava de honestidade VERDE: só inferência neste notebook


## 9. Rolante 2025 por H (janelas viáveis cheias, sem stride)

Acumulação por chunk (4096 janelas): pisos + 9 seed-previsões por família →
seed-means → média-simples-diagnóstico. Rolante acumula Σ|erro| e Σerro² por
(modelo, variável); âncoras e dias-exemplo guardam o MAE por janela.

In [11]:
H = 12
t_H = time.time()
WLN_H = sliding_window_view(Vz["m2"][H], LN, axis=0)   # só p/ monta_ci (M2)
vv = np.arange(P[H]["n"])
C = len(vv)
CHUNK = 4096
ACC = {}    # (modelo, var) -> [sum_abs, sum_sq]
ANC = []    # linhas por âncora: dict(H, data, variavel, modelo -> MAE)
EX = {}     # pos-âncora -> {data, truth (H,2), preds {modelo: (H,2)}}
_is_anc = np.zeros(C, dtype=bool)
_is_anc[P[H]["anchors"]] = True
_ex_set = set(int(k) for k in P[H]["examples"])

def _acc(modelo, var, err):
    k = (modelo, var)
    if k not in ACC:
        ACC[k] = [0.0, 0.0]
    ACC[k][0] += float(np.abs(err).sum())
    ACC[k][1] += float((err ** 2).sum())

for _a in range(0, C, CHUNK):
    _sl = vv[_a:_a + CHUNK]
    _S = P[H]["starts"][_sl]
    _Y = true_chunk(_S, H)                      # (c,H,4) original
    _Yph, _Yod = _Y[:, :, 1], _Y[:, :, 0]
    _F = floor_chunk(_S, H)
    _pred = {"sazonal-naive-288": {"ph": _F["sazonal-naive-288"][:, :, 1],
                                   "od": _F["sazonal-naive-288"][:, :, 0]},
             "persistencia": {"ph": _F["persistencia"][:, :, 1],
                              "od": _F["persistencia"][:, :, 0]}}
    _seeds = {}
    for _fam in FAMS:
        _sm, _ps = preve_fam(_fam, H, _sl, WLN_H)
        _mn = FAM_NOME[_fam]
        _pred[_mn] = {"ph": _sm[:, :, 0], "od": _sm[:, :, 1]}
        _seeds[_mn] = [{"ph": p[:, :, 0], "od": p[:, :, 1]} for p in _ps]
    _ms = {"ph": np.mean([_pred[FAM_NOME[f]]["ph"] for f in FAMS], axis=0),
           "od": np.mean([_pred[FAM_NOME[f]]["od"] for f in FAMS], axis=0)}
    _pred["media-simples"] = _ms
    for _m, _d in _pred.items():
        _acc(_m, "ph", _d["ph"] - _Yph)
        _acc(_m, "od", _d["od"] - _Yod)
    for _j, _mn in enumerate([FAM_NOME[f] for f in FAMS]):
        for _sdi, _sd in enumerate(SEEDS):
            _acc("%s_s%d" % (_mn, _sd), "ph", _seeds[_mn][_sdi]["ph"] - _Yph)
            _acc("%s_s%d" % (_mn, _sd), "od", _seeds[_mn][_sdi]["od"] - _Yod)
    _loc = np.where(_is_anc[_sl])[0]
    for _g in _loc:
        _p = int(_sl[_g])
        _row = {"pos": _p, "data": str(P[H]["ends"][_p].date())}
        for _m, _d in _pred.items():
            _row["%s::ph" % _m] = float(np.abs(_d["ph"][_g] - _Yph[_g]).mean())
            _row["%s::od" % _m] = float(np.abs(_d["od"][_g] - _Yod[_g]).mean())
        for _mn in [FAM_NOME[f] for f in FAMS]:
            for _sdi, _sd in enumerate(SEEDS):
                _row["%s_s%d::ph" % (_mn, _sd)] = float(
                    np.abs(_seeds[_mn][_sdi]["ph"][_g] - _Yph[_g]).mean())
                _row["%s_s%d::od" % (_mn, _sd)] = float(
                    np.abs(_seeds[_mn][_sdi]["od"][_g] - _Yod[_g]).mean())
        ANC.append(_row)
    for _g in range(len(_sl)):
        _p = int(_sl[_g])
        if _p in _ex_set:
            EX[_p] = {"data": str(P[H]["ends"][_p].date()),
                      "truth": np.stack([_Yph[_g], _Yod[_g]], axis=1),
                      "preds": {m: np.stack([d["ph"][_g], d["od"][_g]], axis=1)
                                for m, d in _pred.items()}}
    del _sl, _S, _Y, _Yph, _Yod, _F, _pred, _seeds, _ms
for _fam in FAMS:
    _descarrega(_fam, H)
del WLN_H
gc.collect()
if DEVICE.type == "cuda":
    torch.cuda.empty_cache()
P[H]["ACC"], P[H]["ANC"], P[H]["EX"] = ACC, ANC, EX
_n = float(C * H)
print("=== H=%d rolante (%d origens) em %.0fs ===" % (H, C, time.time() - t_H))
for _var in ("ph", "od"):
    _lin = "  %s " % _var + " | ".join(
        "%s MAE=%.4f RMSE=%.4f" % (_m, ACC[(_m, _var)][0] / _n,
                                   float(np.sqrt(ACC[(_m, _var)][1] / _n)))
        for _m in ("sazonal-naive-288", "dlinear-multi", "patchtst-CI",
                   "patchtst-CD", "media-simples"))
    print(_lin)
print("âncoras guardadas: %d | exemplos: %d" % (len(ANC), len(EX)))


=== H=12 rolante (74752 origens) em 114s ===
  ph sazonal-naive-288 MAE=0.0563 RMSE=0.0786 | dlinear-multi MAE=0.0326 RMSE=0.0454 | patchtst-CI MAE=0.0298 RMSE=0.0421 | patchtst-CD MAE=0.0301 RMSE=0.0423 | media-simples MAE=0.0297 RMSE=0.0418
  od sazonal-naive-288 MAE=0.2486 RMSE=0.3696 | dlinear-multi MAE=0.0773 RMSE=0.1133 | patchtst-CI MAE=0.0305 RMSE=0.0589 | patchtst-CD MAE=0.0343 RMSE=0.0617 | media-simples MAE=0.0380 RMSE=0.0664
âncoras guardadas: 261 | exemplos: 3


In [12]:
H = 72
t_H = time.time()
WLN_H = sliding_window_view(Vz["m2"][H], LN, axis=0)   # só p/ monta_ci (M2)
vv = np.arange(P[H]["n"])
C = len(vv)
CHUNK = 4096
ACC = {}    # (modelo, var) -> [sum_abs, sum_sq]
ANC = []    # linhas por âncora: dict(H, data, variavel, modelo -> MAE)
EX = {}     # pos-âncora -> {data, truth (H,2), preds {modelo: (H,2)}}
_is_anc = np.zeros(C, dtype=bool)
_is_anc[P[H]["anchors"]] = True
_ex_set = set(int(k) for k in P[H]["examples"])

def _acc(modelo, var, err):
    k = (modelo, var)
    if k not in ACC:
        ACC[k] = [0.0, 0.0]
    ACC[k][0] += float(np.abs(err).sum())
    ACC[k][1] += float((err ** 2).sum())

for _a in range(0, C, CHUNK):
    _sl = vv[_a:_a + CHUNK]
    _S = P[H]["starts"][_sl]
    _Y = true_chunk(_S, H)                      # (c,H,4) original
    _Yph, _Yod = _Y[:, :, 1], _Y[:, :, 0]
    _F = floor_chunk(_S, H)
    _pred = {"sazonal-naive-288": {"ph": _F["sazonal-naive-288"][:, :, 1],
                                   "od": _F["sazonal-naive-288"][:, :, 0]},
             "persistencia": {"ph": _F["persistencia"][:, :, 1],
                              "od": _F["persistencia"][:, :, 0]}}
    _seeds = {}
    for _fam in FAMS:
        _sm, _ps = preve_fam(_fam, H, _sl, WLN_H)
        _mn = FAM_NOME[_fam]
        _pred[_mn] = {"ph": _sm[:, :, 0], "od": _sm[:, :, 1]}
        _seeds[_mn] = [{"ph": p[:, :, 0], "od": p[:, :, 1]} for p in _ps]
    _ms = {"ph": np.mean([_pred[FAM_NOME[f]]["ph"] for f in FAMS], axis=0),
           "od": np.mean([_pred[FAM_NOME[f]]["od"] for f in FAMS], axis=0)}
    _pred["media-simples"] = _ms
    for _m, _d in _pred.items():
        _acc(_m, "ph", _d["ph"] - _Yph)
        _acc(_m, "od", _d["od"] - _Yod)
    for _j, _mn in enumerate([FAM_NOME[f] for f in FAMS]):
        for _sdi, _sd in enumerate(SEEDS):
            _acc("%s_s%d" % (_mn, _sd), "ph", _seeds[_mn][_sdi]["ph"] - _Yph)
            _acc("%s_s%d" % (_mn, _sd), "od", _seeds[_mn][_sdi]["od"] - _Yod)
    _loc = np.where(_is_anc[_sl])[0]
    for _g in _loc:
        _p = int(_sl[_g])
        _row = {"pos": _p, "data": str(P[H]["ends"][_p].date())}
        for _m, _d in _pred.items():
            _row["%s::ph" % _m] = float(np.abs(_d["ph"][_g] - _Yph[_g]).mean())
            _row["%s::od" % _m] = float(np.abs(_d["od"][_g] - _Yod[_g]).mean())
        for _mn in [FAM_NOME[f] for f in FAMS]:
            for _sdi, _sd in enumerate(SEEDS):
                _row["%s_s%d::ph" % (_mn, _sd)] = float(
                    np.abs(_seeds[_mn][_sdi]["ph"][_g] - _Yph[_g]).mean())
                _row["%s_s%d::od" % (_mn, _sd)] = float(
                    np.abs(_seeds[_mn][_sdi]["od"][_g] - _Yod[_g]).mean())
        ANC.append(_row)
    for _g in range(len(_sl)):
        _p = int(_sl[_g])
        if _p in _ex_set:
            EX[_p] = {"data": str(P[H]["ends"][_p].date()),
                      "truth": np.stack([_Yph[_g], _Yod[_g]], axis=1),
                      "preds": {m: np.stack([d["ph"][_g], d["od"][_g]], axis=1)
                                for m, d in _pred.items()}}
    del _sl, _S, _Y, _Yph, _Yod, _F, _pred, _seeds, _ms
for _fam in FAMS:
    _descarrega(_fam, H)
del WLN_H
gc.collect()
if DEVICE.type == "cuda":
    torch.cuda.empty_cache()
P[H]["ACC"], P[H]["ANC"], P[H]["EX"] = ACC, ANC, EX
_n = float(C * H)
print("=== H=%d rolante (%d origens) em %.0fs ===" % (H, C, time.time() - t_H))
for _var in ("ph", "od"):
    _lin = "  %s " % _var + " | ".join(
        "%s MAE=%.4f RMSE=%.4f" % (_m, ACC[(_m, _var)][0] / _n,
                                   float(np.sqrt(ACC[(_m, _var)][1] / _n)))
        for _m in ("sazonal-naive-288", "dlinear-multi", "patchtst-CI",
                   "patchtst-CD", "media-simples"))
    print(_lin)
print("âncoras guardadas: %d | exemplos: %d" % (len(ANC), len(EX)))


=== H=72 rolante (74032 origens) em 114s ===
  ph sazonal-naive-288 MAE=0.0563 RMSE=0.0787 | dlinear-multi MAE=0.0391 RMSE=0.0544 | patchtst-CI MAE=0.0367 RMSE=0.0515 | patchtst-CD MAE=0.0381 RMSE=0.0527 | media-simples MAE=0.0360 RMSE=0.0504
  od sazonal-naive-288 MAE=0.2480 RMSE=0.3688 | dlinear-multi MAE=0.1436 RMSE=0.2111 | patchtst-CI MAE=0.1194 RMSE=0.1865 | patchtst-CD MAE=0.1365 RMSE=0.2033 | media-simples MAE=0.1158 RMSE=0.1816
âncoras guardadas: 260 | exemplos: 3


In [13]:
H = 288
t_H = time.time()
WLN_H = sliding_window_view(Vz["m2"][H], LN, axis=0)   # só p/ monta_ci (M2)
vv = np.arange(P[H]["n"])
C = len(vv)
CHUNK = 4096
ACC = {}    # (modelo, var) -> [sum_abs, sum_sq]
ANC = []    # linhas por âncora: dict(H, data, variavel, modelo -> MAE)
EX = {}     # pos-âncora -> {data, truth (H,2), preds {modelo: (H,2)}}
_is_anc = np.zeros(C, dtype=bool)
_is_anc[P[H]["anchors"]] = True
_ex_set = set(int(k) for k in P[H]["examples"])

def _acc(modelo, var, err):
    k = (modelo, var)
    if k not in ACC:
        ACC[k] = [0.0, 0.0]
    ACC[k][0] += float(np.abs(err).sum())
    ACC[k][1] += float((err ** 2).sum())

for _a in range(0, C, CHUNK):
    _sl = vv[_a:_a + CHUNK]
    _S = P[H]["starts"][_sl]
    _Y = true_chunk(_S, H)                      # (c,H,4) original
    _Yph, _Yod = _Y[:, :, 1], _Y[:, :, 0]
    _F = floor_chunk(_S, H)
    _pred = {"sazonal-naive-288": {"ph": _F["sazonal-naive-288"][:, :, 1],
                                   "od": _F["sazonal-naive-288"][:, :, 0]},
             "persistencia": {"ph": _F["persistencia"][:, :, 1],
                              "od": _F["persistencia"][:, :, 0]}}
    _seeds = {}
    for _fam in FAMS:
        _sm, _ps = preve_fam(_fam, H, _sl, WLN_H)
        _mn = FAM_NOME[_fam]
        _pred[_mn] = {"ph": _sm[:, :, 0], "od": _sm[:, :, 1]}
        _seeds[_mn] = [{"ph": p[:, :, 0], "od": p[:, :, 1]} for p in _ps]
    _ms = {"ph": np.mean([_pred[FAM_NOME[f]]["ph"] for f in FAMS], axis=0),
           "od": np.mean([_pred[FAM_NOME[f]]["od"] for f in FAMS], axis=0)}
    _pred["media-simples"] = _ms
    for _m, _d in _pred.items():
        _acc(_m, "ph", _d["ph"] - _Yph)
        _acc(_m, "od", _d["od"] - _Yod)
    for _j, _mn in enumerate([FAM_NOME[f] for f in FAMS]):
        for _sdi, _sd in enumerate(SEEDS):
            _acc("%s_s%d" % (_mn, _sd), "ph", _seeds[_mn][_sdi]["ph"] - _Yph)
            _acc("%s_s%d" % (_mn, _sd), "od", _seeds[_mn][_sdi]["od"] - _Yod)
    _loc = np.where(_is_anc[_sl])[0]
    for _g in _loc:
        _p = int(_sl[_g])
        _row = {"pos": _p, "data": str(P[H]["ends"][_p].date())}
        for _m, _d in _pred.items():
            _row["%s::ph" % _m] = float(np.abs(_d["ph"][_g] - _Yph[_g]).mean())
            _row["%s::od" % _m] = float(np.abs(_d["od"][_g] - _Yod[_g]).mean())
        for _mn in [FAM_NOME[f] for f in FAMS]:
            for _sdi, _sd in enumerate(SEEDS):
                _row["%s_s%d::ph" % (_mn, _sd)] = float(
                    np.abs(_seeds[_mn][_sdi]["ph"][_g] - _Yph[_g]).mean())
                _row["%s_s%d::od" % (_mn, _sd)] = float(
                    np.abs(_seeds[_mn][_sdi]["od"][_g] - _Yod[_g]).mean())
        ANC.append(_row)
    for _g in range(len(_sl)):
        _p = int(_sl[_g])
        if _p in _ex_set:
            EX[_p] = {"data": str(P[H]["ends"][_p].date()),
                      "truth": np.stack([_Yph[_g], _Yod[_g]], axis=1),
                      "preds": {m: np.stack([d["ph"][_g], d["od"][_g]], axis=1)
                                for m, d in _pred.items()}}
    del _sl, _S, _Y, _Yph, _Yod, _F, _pred, _seeds, _ms
for _fam in FAMS:
    _descarrega(_fam, H)
del WLN_H
gc.collect()
if DEVICE.type == "cuda":
    torch.cuda.empty_cache()
P[H]["ACC"], P[H]["ANC"], P[H]["EX"] = ACC, ANC, EX
_n = float(C * H)
print("=== H=%d rolante (%d origens) em %.0fs ===" % (H, C, time.time() - t_H))
for _var in ("ph", "od"):
    _lin = "  %s " % _var + " | ".join(
        "%s MAE=%.4f RMSE=%.4f" % (_m, ACC[(_m, _var)][0] / _n,
                                   float(np.sqrt(ACC[(_m, _var)][1] / _n)))
        for _m in ("sazonal-naive-288", "dlinear-multi", "patchtst-CI",
                   "patchtst-CD", "media-simples"))
    print(_lin)
print("âncoras guardadas: %d | exemplos: %d" % (len(ANC), len(EX)))


=== H=288 rolante (71607 origens) em 99s ===
  ph sazonal-naive-288 MAE=0.0565 RMSE=0.0789 | dlinear-multi MAE=0.0475 RMSE=0.0657 | patchtst-CI MAE=0.0491 RMSE=0.0679 | patchtst-CD MAE=0.0493 RMSE=0.0671 | media-simples MAE=0.0454 RMSE=0.0629
  od sazonal-naive-288 MAE=0.2473 RMSE=0.3687 | dlinear-multi MAE=0.2305 RMSE=0.3265 | patchtst-CI MAE=0.2410 RMSE=0.3438 | patchtst-CD MAE=0.2498 RMSE=0.3454 | media-simples MAE=0.2160 RMSE=0.3099
âncoras guardadas: 250 | exemplos: 3


## 10. Tabelas (rolante + dias-âncora + por mês)

In [14]:
MODELOS_TBL = ["sazonal-naive-288", "persistencia", "dlinear-multi", "patchtst-CI",
               "patchtst-CD", "media-simples"]
rows_b, rows_d, rows_m = [], [], []
for H in HS:
    n = float(P[H]["n"] * H)
    for var in ("ph", "od"):
        for m in MODELOS_TBL:
            sa, sq = P[H]["ACC"][(m, var)]
            rows_b.append({"H": H, "variavel": var, "modelo": m, "seed": "seed-mean/fixo",
                           "MAE": round(sa / n, 4), "RMSE": round(float(np.sqrt(sq / n)), 4),
                           "n_origens": P[H]["n"]})
        for fam_nome in [FAM_NOME[f] for f in FAMS]:
            mas, mds, rss = [], [], []
            for sd in SEEDS:
                sa, sq = P[H]["ACC"][("%s_s%d" % (fam_nome, sd), var)]
                a, r = sa / n, float(np.sqrt(sq / n))
                rows_b.append({"H": H, "variavel": var, "modelo": fam_nome, "seed": sd,
                               "MAE": round(a, 4), "RMSE": round(r, 4), "n_origens": P[H]["n"]})
                mas.append(a); rss.append(r)
            P[H].setdefault("seedstats", {})[(fam_nome, var)] = (
                float(np.mean(mas)), float(np.std(mas, ddof=1)),
                float(np.mean(rss)), float(np.std(rss, ddof=1)))
    anc = pd.DataFrame(P[H]["ANC"]).sort_values("pos").reset_index(drop=True)
    assert len(anc) == len(P[H]["anchors"]), (H, len(anc), len(P[H]["anchors"]))
    for _, r in anc.iterrows():
        for var in ("ph", "od"):
            d = {"H": H, "data": r["data"], "variavel": var}
            for m in MODELOS_TBL:
                d[m] = round(float(r["%s::%s" % (m, var)]), 4)
            for fam_nome in [FAM_NOME[f] for f in FAMS]:
                arr = [float(r["%s_s%d::%s" % (fam_nome, sd, var)]) for sd in SEEDS]
                for sd, a in zip(SEEDS, arr):
                    d["%s_s%d" % (fam_nome, sd)] = round(a, 4)
                d["%s_media" % fam_nome] = round(float(np.mean(arr)), 4)
                d["%s_dp" % fam_nome] = round(float(np.std(arr, ddof=1)), 4)
            rows_d.append(d)
    dd = pd.DataFrame(rows_d[-len(anc) * 2:])
    dd["mes"] = pd.to_datetime(dd["data"]).dt.month
    nomes_mes = ["jan", "fev", "mar", "abr", "mai", "jun",
                 "jul", "ago", "set", "out", "nov", "dez"]
    for (m, var), g in dd.groupby(["mes", "variavel"]):
        r = {"H": H, "mes": nomes_mes[m - 1], "variavel": var, "n_dias": len(g)}
        for c in MODELOS_TBL + ["%s_media" % FAM_NOME[f] for f in FAMS]:
            r[c] = round(float(g[c].mean()), 4)
        rows_m.append(r)

tab_b = pd.DataFrame(rows_b)
tab_b.to_csv(OUT / "metricas_benchmark.csv", index=False)
tab_d = pd.DataFrame(rows_d)
tab_d.to_csv(OUT / "metricas_diaria.csv", index=False)
tab_m = pd.DataFrame(rows_m)
tab_m.to_csv(OUT / "metricas_por_mes.csv", index=False)
print("metricas_benchmark:", tab_b.shape, "| metricas_diaria:", tab_d.shape,
      "| metricas_por_mes:", tab_m.shape)
assert len(tab_b) == sum(2 * (len(MODELOS_TBL) + 3 * len(SEEDS)) for H in HS), len(tab_b)
assert tab_d[["H", "data", "variavel"]].duplicated().sum() == 0
print("=== rolante 2025 (seed-mean / pisos fixos) ===")
piv = tab_b[tab_b["seed"] == "seed-mean/fixo"].pivot_table(
    index=["H", "variavel"], columns="modelo", values="MAE")[MODELOS_TBL]
print(piv.round(4).to_string())
print("=== dias-âncora: média por (H, var) (checagem, não manchete) ===")
print(tab_d.groupby(["H", "variavel"])[MODELOS_TBL].mean().round(4).to_string())
print("=== MAE médio por mês (âncoras) ===")
print(tab_m.pivot_table(index=["H", "mes"], columns="variavel",
                        values="media-simples").round(4).to_string())

metricas_benchmark: (90, 7) | metricas_diaria: (1542, 24) | metricas_por_mes: (72, 13)
=== rolante 2025 (seed-mean / pisos fixos) ===
modelo        sazonal-naive-288  persistencia  dlinear-multi  patchtst-CI  patchtst-CD  media-simples
H   variavel                                                                                         
12  od                   0.2486        0.0599         0.0773       0.0305       0.0343         0.0380
    ph                   0.0563        0.0368         0.0326       0.0298       0.0301         0.0297
72  od                   0.2480        0.3043         0.1436       0.1194       0.1365         0.1158
    ph                   0.0563        0.0540         0.0391       0.0367       0.0381         0.0360
288 od                   0.2473        0.5340         0.2305       0.2410       0.2498         0.2160
    ph                   0.0565        0.0751         0.0475       0.0491       0.0493         0.0454
=== dias-âncora: média por (H, var) (checagem, não

## 11. Figuras (barras MAE por (H,var), erro mensal, 3 dias-exemplo por H)

In [15]:
CORES = {"sazonal-naive-288": "gray", "persistencia": "lightgray", "dlinear-multi": "C0",
         "patchtst-CI": "C1", "patchtst-CD": "C2", "media-simples": "k"}
for H in HS:
    gb = tab_b[(tab_b["H"] == H) & (tab_b["seed"] == "seed-mean/fixo")].set_index(["variavel", "modelo"])
    fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
    for ax, var in zip(axes, ["ph", "od"]):
        ys = [float(gb.loc[(var, m), "MAE"]) for m in MODELOS_TBL]
        dp = [float(P[H]["seedstats"][(m, var)][1]) if (m, var) in P[H].get("seedstats", {}) else 0.0
              for m in MODELOS_TBL]
        ax.bar(MODELOS_TBL, ys, yerr=dp, capsize=4,
               color=[CORES[m] for m in MODELOS_TBL])
        ax.set_title("%s H=%d — MAE rolante 2025 (menor = melhor)" % (var, H))
        ax.tick_params(axis="x", rotation=18, labelsize=8)
        for x, y in zip(MODELOS_TBL, ys):
            ax.text(x, y, "%.4f" % y, ha="center", va="bottom", fontsize=8)
    fig.tight_layout(); fig.savefig(OUT / "figs" / ("04-mae-H%d.png" % H)); plt.close(fig)

    gm = tab_m[tab_m["H"] == H].copy()
    ordem = ["jan", "fev", "mar", "abr", "mai", "jun",
             "jul", "ago", "set", "out", "nov", "dez"]
    gm["mes"] = pd.Categorical(gm["mes"], ordem, ordered=True)
    fig, axes = plt.subplots(2, 1, figsize=(13, 7), sharex=True)
    for ax, var in zip(axes, ["ph", "od"]):
        g = gm[gm["variavel"] == var].sort_values("mes")
        for m in MODELOS_TBL:
            ax.plot(g["mes"].astype(str), g[m], marker="o", ms=3, lw=1.2, label=m,
                    color=CORES[m])
        ax.set_title("%s H=%d — MAE médio por mês em 2025 (sazonalidade do erro)" % (var, H))
        ax.legend(fontsize=8)
    fig.tight_layout(); fig.savefig(OUT / "figs" / ("05-mensal-H%d.png" % H)); plt.close(fig)

    fig, axes = plt.subplots(3, 2, figsize=(14, 10), sharex=False)
    for ax_row, _p in zip(axes, sorted(P[H]["EX"])):
        e = P[H]["EX"][_p]
        tf = pd.date_range(pd.Timestamp(e["data"]) - pd.Timedelta(minutes=5 * (H - 1)),
                           pd.Timestamp(e["data"]), freq="5min")
        for ax, vi, var in zip(ax_row, [0, 1], ["ph", "od"]):
            ax.plot(tf, e["truth"][:, vi], "k-", lw=1.2, label="real")
            ax.plot(tf, e["preds"]["sazonal-naive-288"][:, vi], ":", lw=1, label="saz-288",
                    color="gray")
            for m, ls in [("dlinear-multi", "--"), ("patchtst-CI", "-"),
                          ("patchtst-CD", "-."), ("media-simples", "-")]:
                ax.plot(tf, e["preds"][m][:, vi], lw=1, ls=ls, alpha=0.9, label=m)
            ax.set_title("%s H=%d dia %s" % (var, H, e["data"]))
            ax.legend(fontsize=7)
    fig.tight_layout(); fig.savefig(OUT / "figs" / ("06-exemplos-H%d.png" % H)); plt.close(fig)
print("figs 04/05/06 salvas (3 por H)")
print("figs totais:", sorted(p.name for p in (OUT / "figs").glob("*.png")))

figs 04/05/06 salvas (3 por H)
figs totais: ['01-eda.png', '02-limpeza.png', '04-mae-H12.png', '04-mae-H288.png', '04-mae-H72.png', '05-mensal-H12.png', '05-mensal-H288.png', '05-mensal-H72.png', '06-exemplos-H12.png', '06-exemplos-H288.png', '06-exemplos-H72.png']


## 12. Veredito + proveniência (números reais impressos abaixo)

Rolante 2025 decide; dias-âncora checam. Comparação honesta: réguas uni-v2 H=288
(pH 0,0465 · OD 0,2056, mesmo ano de teste, val de origem distinta) e pisos multi
deste benchmark. Sem declarar régua nova sem teste de margem (±dp entre seeds).

In [16]:
print("==================== M4 RESUMO FINAL ====================")
print("--- cobertura 2025 por H ---")
for H in HS:
    print("H=%3d janelas viáveis=%d | dias-âncora=%d (%s -> %s)" % (
        H, P[H]["n"], len(P[H]["anchors"]),
        P[H]["ends"][P[H]["anchors"][0]].date(), P[H]["ends"][P[H]["anchors"][-1]].date()))
print("--- rolante 2025 MAE (seed-mean / pisos) ---")
piv = tab_b[tab_b["seed"] == "seed-mean/fixo"].pivot_table(
    index=["H", "variavel"], columns="modelo", values="MAE")[MODELOS_TBL]
print(piv.round(4).to_string())
print("--- ±dp entre seeds no rolante ---")
for H in HS:
    for var in ("ph", "od"):
        print("H=%3d %s " % (H, var) + " | ".join(
            "%s %.4f±%.4f" % (FAM_NOME[f], *P[H]["seedstats"][(FAM_NOME[f], var)][:2])
            for f in FAMS))
print("--- val 2024 (M1/M2/M3 pooled, seed-mean) × 2025 (este bench), MAE ---")
val = {}
for f, pre in [("m1", "dl"), ("m2", "ci"), ("m3", "cd")]:
    t = pd.read_csv(ROOT / "multivariavel" / "resultados" /
                    ({"m1": "M1-dlinear-multi", "m2": "M2-patchtst-multi-CI",
                      "m3": "M3-patchtst-multi-CD"}[f]) / "metricas_pooled.csv").set_index("H")
    for H in HS:
        for var in ("ph", "od"):
            val[(H, var, FAM_NOME[f])] = (
                float(t.loc[H, "%s_%s_MAE_media" % (pre, var)]),
                float(t.loc[H, "%s_%s_MAE_dp" % (pre, var)]))
for H in HS:
    for var in ("ph", "od"):
        print("H=%3d %s " % (H, var) + " | ".join(
            "%s val %.4f±%.4f -> 2025 %.4f±%.4f" % (
                FAM_NOME[f], *val[(H, var, FAM_NOME[f])],
                *P[H]["seedstats"][(FAM_NOME[f], var)][:2]) for f in FAMS))
print("--- H=288 × réguas uni-v2 (mesmo ano de teste; val de origem distinta) ---")
for var in ("ph", "od"):
    print("%s régua uni-v2: %.4f | " % (var, REGUA_UNI[var]) + " | ".join(
        "multi-%s %.4f±%.4f" % (FAM_NOME[f], *P[288]["seedstats"][(FAM_NOME[f], var)][:2])
        for f in FAMS))
print("---- arquivos gerados ----")
for p in sorted(OUT.rglob("*")):
    if p.is_file():
        print(" ", p.relative_to(ROOT), "(%.1f KB)" % (p.stat().st_size / 1024))
for f in ["metricas_benchmark.csv", "metricas_diaria.csv", "metricas_por_mes.csv"]:
    assert (OUT / f).exists(), f
assert len(list((OUT / "figs").glob("*.png"))) == 11, "figs faltando!"
print("0-error/27-ckpts/3-CSVs/11-figs: asserts verdes")
import subprocess as _sp
try:
    _head = _sp.check_output(["git", "rev-parse", "--short", "HEAD"], cwd=ROOT, text=True).strip()
except Exception:
    _head = "<sem git>"
print("---- proveniência ----")
print("host:", socket.gethostname(), "| cpu:", os.cpu_count(), "| torch:", torch.__version__,
      "| device:", DEVICE, "| git HEAD:", _head)
if DEVICE.type == "cuda":
    print("gpu:", torch.cuda.get_device_name(0))
print("M4_DEVICE=", os.environ.get("M4_DEVICE", "<unset>"),
      "CUDA_VISIBLE_DEVICES=", os.environ.get("CUDA_VISIBLE_DEVICES", "<unset>"))
print("2025 intocado até este notebook: só inferência, sem gradiente, sem ajuste em 2025")
print("reprodução: CUDA_VISIBLE_DEVICES=0 M4_DEVICE=cuda .venv/bin/jupyter nbconvert "
      "--to notebook --execute --inplace --ExecutePreprocessor.timeout=7200 "
      "multivariavel/notebooks/M4-benchmark-2025.ipynb")
print("wall time total: %.1f min" % ((time.time() - t_wall0) / 60))

==================== M4 RESUMO FINAL ====================
--- cobertura 2025 por H ---
H= 12 janelas viáveis=74752 | dias-âncora=261 (2025-01-09 -> 2025-12-30)
H= 72 janelas viáveis=74032 | dias-âncora=260 (2025-01-09 -> 2025-12-30)
H=288 janelas viáveis=71607 | dias-âncora=250 (2025-01-09 -> 2025-12-30)
--- rolante 2025 MAE (seed-mean / pisos) ---
modelo        sazonal-naive-288  persistencia  dlinear-multi  patchtst-CI  patchtst-CD  media-simples
H   variavel                                                                                         
12  od                   0.2486        0.0599         0.0773       0.0305       0.0343         0.0380
    ph                   0.0563        0.0368         0.0326       0.0298       0.0301         0.0297
72  od                   0.2480        0.3043         0.1436       0.1194       0.1365         0.1158
    ph                   0.0563        0.0540         0.0391       0.0367       0.0381         0.0360
288 od                   0.2473      